# Train BandNet Mobile1D (176 × 512) in Google Colab

Select **Runtime → Change runtime type → GPU**, then **Run all**. The notebook automatically mounts Google Drive and trains from the three public `r1` COCO datasets.

### Persistent dataset cache
The prepared dataset (extracted images, canonical annotations, leakage-safe splits) is cached as one checksummed ZIP plus a manifest under `MyDrive/resistor_model/r1-mobile1d-colab/`. After its first creation, new Colab sessions restore it instead of downloading and preparing the datasets again. Each original release ZIP is also verified against its GitHub SHA-256 and stored in `archives/` for recovery.

### Checkpointing and progress
- `last.pt` saved after each completed epoch; training resumes automatically after Colab disconnects.
- `best.pt` updated when validation exact-sequence accuracy / macro-F1 improves.
- `checkpoints/epoch_005.pt`, `epoch_010.pt`, etc. contain additional recovery snapshots.
- Console reports the current stage, download/extraction progress, train/validation batch counts, percentage, loss, LR, elapsed time and ETA. Silent training operations print a heartbeat approximately every 30 seconds.
- The preflight runs Mobile1D, dataset, training and cache tests only, so unrelated broken segmentation/detector tests cannot prevent the training run.

### Outputs
The model input is **176×512 (H×W)** with **256 sequence bins**. The trained outputs `best.pt`, `resistor_bandnet_mobile1d.onnx`, its `.onnx.json` metadata, and best-effort `resistor_bandnet_mobile1d.tflite` appear in `MyDrive/resistor_model/r1-mobile1d-colab/`. The final browser ZIP contains **model/checkpoint/metric outputs only**, not the large cached datasets.

Optional V2 teacher distillation uses `MyDrive/resistor_model/r1-v2-colab/best.pt` if that file exists; a folder alone does not establish a teacher checkpoint.

### 120-epoch continuation
The run now targets **120 epochs** instead of 60. Epochs 1–60 retain the original schedule, while epochs 61–120 use a **new, lower-rate cosine cycle** starting at 20% of the initial peak LR. Re-running the updated launcher resumes `last.pt` from Drive (including after a finished 60-epoch run), preserves `best.pt`, and avoids an unintentional full-rate jump. An already-running old 60-epoch process will still finish at 60; restart the updated launcher after it exits to continue.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# Use a fresh Git checkout instead of cached raw.githubusercontent.com files.
ref = "feat/mobile1d-colab-progress-drive-cache"
os.environ["MOBILE1D_GIT_REF"] = ref
repo = Path("/content/mobile1d_script_source")
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    "git", "clone", "--depth", "1", "--single-branch", "--branch", ref,
    "https://github.com/Persie0/resistor_model.git", str(repo),
], check=True)
commit = subprocess.check_output([
    "git", "-C", str(repo), "rev-parse", "HEAD"
], text=True).strip()
source_path = repo / "colab/train_rres_mobile1d_colab.py"
source = source_path.read_bytes()
expected = b"2026-10-16-v4-120-epoch-continuation"
if expected not in source:
    raise RuntimeError(f"Checked out an old script at {commit}; update the branch or use the latest PR commit.")
print(f"[launcher] checked out {commit[:12]} | verified v3 script", flush=True)
exec(compile(source, str(source_path), "exec"), {"__name__": "__main__"})
